# 🎵 MoodBeats AI — Session 8: Deployment & Showcase

**Today your app goes live.** By the end of this session you will have a public URL to share with friends and family.

---
## The Journey So Far

| Session | What we built |
|---|---|
| 1 | Python basics — variables, functions, if/else |
| 2 | Lists, dicts — the data structures powering MoodBeats |
| 3 | Gemini API — AI mood detection from text |
| 4 | FastAPI + Jinja2 — the web server and templates |
| 5 | Detect page — POST-Redirect-GET pattern |
| 6 | Music links + results page — enrich() and themes |
| 7 | Sessions + Supabase — memory and history |
| **8** | **Git + Render deployment — go live!** |

---

## Part 1: Environment Variables — Never Hardcode Secrets

**Rule:** API keys and passwords must NEVER appear in your code or git history.

Instead:
- Locally: store in a `.env` file (never commit this file)
- On Render: set in the dashboard under "Environment"

Python reads them with `os.getenv()`.

In [ ]:
import os

# This is how the real app reads env vars
# On Render, these are set in the dashboard.
# Locally, they're loaded from a .env file via python-dotenv.

gemini_key = os.getenv("GEMINI_API_KEY", "NOT_SET")
secret_key = os.getenv("SECRET_KEY",     "dev-secret-change-me")
supabase_url = os.getenv("SUPABASE_URL", "")

print(f"GEMINI_API_KEY: {'SET ✓' if gemini_key != 'NOT_SET' else 'NOT SET ✗'}")
print(f"SECRET_KEY:     {'SET ✓' if secret_key != 'dev-secret-change-me' else 'dev default'}")  
print(f"SUPABASE_URL:   {'SET ✓' if supabase_url else 'not configured (optional)'})")

In [ ]:
# The .env file format (stored locally, never committed to git)
env_example_content = """# .env — Copy to .env and fill in your values
# NEVER commit this file to git!

GEMINI_API_KEY=your_gemini_api_key_here
SECRET_KEY=any-long-random-string-you-make-up
SUPABASE_URL=https://your-project.supabase.co
SUPABASE_KEY=your_supabase_anon_key_here
"""

print(".env.example contents:")
print(env_example_content)
print("This file is already in the MoodBeats AI project as .env.example")

---
## Part 2: Git — Version Control Basics

Git tracks every change to your code. GitHub stores it online. Render deploys from GitHub.

```
Your computer → git push → GitHub → Render auto-deploys
```

### Essential Git Commands

```bash
git init                    # Start tracking this folder
git add .                   # Stage all changes
git add app.py              # Stage one file
git status                  # See what's staged
git commit -m "Add feature" # Save a snapshot with a message
git log                     # See history
git push origin main        # Upload to GitHub
```

In [ ]:
# Run these in a TERMINAL (not Colab) in your MoodBeats AI project folder
# This cell just prints the commands for reference

commands = [
    ("Create .gitignore",    "echo '.env\n__pycache__\n*.pyc\n.DS_Store' > .gitignore"),
    ("Initialise git",       "git init"),
    ("Stage all files",      "git add ."),
    ("First commit",         'git commit -m "Initial commit: MoodBeats AI"'),
    ("Create GitHub repo",   "gh repo create moodbeats-ai --public --source=. --push"),
    ("(or manually push)",   "git remote add origin https://github.com/YOUR_USERNAME/moodbeats-ai.git"),
    ("",                     "git push -u origin main"),
]

print("Run these commands in your terminal:")
print("=" * 60)
for label, cmd in commands:
    if label:
        print(f"\n# {label}")
    print(f"$ {cmd}")

---
## Part 3: The .gitignore File

`.gitignore` tells git which files to NEVER track (secrets, compiled files, etc.).

In [ ]:
gitignore_content = """# Environment variables — NEVER commit these
.env

# Python compiled files
__pycache__/
*.pyc
*.pyo
*.pyd

# Virtual environments
venv/
.venv/

# IDE files
.vscode/
.idea/
.DS_Store

# Logs
*.log
"""

print(".gitignore contents (create this file in your project root):")
print(gitignore_content)

### ✏️ In-Class Exercise 8a — Verify Your .gitignore

In [ ]:
# Test that .gitignore works correctly
# After git add ., run: git status
# The .env file should NOT appear in the staged files list.

# If .env appears, you forgot to create .gitignore first!
# Fix: git rm --cached .env    (removes it from staging)

print("Checklist before committing:")
checklist = [
    "[ ] .gitignore is created and contains .env",
    "[ ] .env file exists locally (for local testing)",
    "[ ] git status shows .env as 'ignored' not 'staged'",
    "[ ] requirements.txt is up to date (pip freeze > requirements.txt)",
    "[ ] render.yaml is present in the project root",
    "[ ] app.py starts FastAPI on the $PORT env var",
]
for item in checklist:
    print(item)

---
## Part 4: Deploying to Render

### Method 1: Using render.yaml (Recommended — already in the project)

1. Go to **https://render.com** and sign up with GitHub
2. Click **New → Blueprint**
3. Connect your `moodbeats-ai` GitHub repo
4. Render reads `render.yaml` and auto-configures everything
5. Add your env vars in the Render dashboard → Environment
6. Click **Deploy**

### Method 2: Manual Web Service

1. Click **New → Web Service**
2. Connect GitHub repo
3. Settings:
   - **Runtime:** Python 3
   - **Build Command:** `pip install -r requirements.txt`
   - **Start Command:** `uvicorn app:app --host 0.0.0.0 --port $PORT`
4. Add env vars: GEMINI_API_KEY, SECRET_KEY, SUPABASE_URL, SUPABASE_KEY

In [ ]:
# The render.yaml file in the project
render_yaml = """
services:
  - type: web
    name: moodbeats-ai
    runtime: python
    plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: uvicorn app:app --host 0.0.0.0 --port $PORT
    envVars:
      - key: GEMINI_API_KEY
        sync: false           # set manually in Render dashboard
      - key: SECRET_KEY
        generateValue: true   # Render auto-generates a secure random value
      - key: SUPABASE_URL
        sync: false
      - key: SUPABASE_KEY
        sync: false
"""
print(render_yaml)

---
## Part 5: Reading Render Build Logs

When a deployment fails, check the **Logs** tab in the Render dashboard.

In [ ]:
# Common deployment errors and how to fix them
errors = [
    (
        "ModuleNotFoundError: No module named 'xyz'",
        "Package 'xyz' is not in requirements.txt. Run: pip freeze > requirements.txt and push."
    ),
    (
        "KeyError: 'GEMINI_API_KEY'",
        "The env var was not set in Render dashboard. Go to your service → Environment → Add GEMINI_API_KEY."
    ),
    (
        "ValueError: GeminiService requires GEMINI_API_KEY",
        "Same as above — the API key is missing."
    ),
    (
        "Error: Cannot bind to port 8000",
        "Change start command to: uvicorn app:app --host 0.0.0.0 --port $PORT (use $PORT, not 8000)"
    ),
    (
        "Build succeeded but site shows 'Service Unavailable'",
        "Check the runtime logs. Usually an import error or missing env var at startup."
    ),
]

print("Common Render errors:")
print("=" * 60)
for error, fix in errors:
    print(f"\n❌ {error}")
    print(f"✅ Fix: {fix}")

---
## Part 6: Verifying Your Live App

Once deployed, run these checks:

In [ ]:
# Replace with your actual Render URL
YOUR_RENDER_URL = "https://your-app-name.onrender.com"  # TODO: update this

checks = [
    ("Landing page loads",       f"{YOUR_RENDER_URL}/"),
    ("Detect page loads",        f"{YOUR_RENDER_URL}/detect"),
    ("History page loads",       f"{YOUR_RENDER_URL}/history"),
    ("Profile page loads",       f"{YOUR_RENDER_URL}/profile"),
    ("404 page works",           f"{YOUR_RENDER_URL}/nonexistent"),
]

import requests as req

print("Running live site checks...")
for label, url in checks:
    try:
        r = req.get(url, timeout=60)  # free tier may be slow to wake
        status = "✅" if r.status_code in (200, 404) else "❌"
        print(f"{status} {label}: HTTP {r.status_code}")
    except Exception as e:
        print(f"❌ {label}: {e}")

---
## 🎉 Showcase Time!

Each student presents their live app:

1. Open your Render URL in the browser
2. Describe your mood in the detect page
3. Show the results — click a Spotify or YouTube link
4. Show your history page

**2 minutes per student. Celebrate every working app!**

---

## 🏠 After-Class Challenges (Ongoing)

Your deployed app is just the beginning. Try these to level up:

---
### Challenge 1 (Easy): Personalise Your Landing Page

In [ ]:
# Ideas to personalise your MoodBeats:
# - Change the app name and tagline in landing.html
# - Change the colour palette (update Tailwind classes or theme colours)
# - Add your name to the footer
# - Change the hero emoji or gradient

# After any change:
#   git add templates/landing.html
#   git commit -m "Personalise landing page"
#   git push origin main
# → Render auto-deploys in ~1-2 minutes

print("Edit templates/landing.html and push to see changes live!")

### Challenge 2 (Medium): Add a New Emotion Theme

In [ ]:
# Open services/music_service.py in the project
# Add a new emotion to EMOTION_THEMES, e.g. 'nostalgic', 'hopeful', 'bored'

new_theme_example = {
    "nostalgic": {
        "accent": "#fbbf24",   # amber
        "bg": "#78350f",       # dark amber
        "border": "#d97706",
        "emoji": "🌅"
    }
}

print("Add this to EMOTION_THEMES in music_service.py")
print(new_theme_example)

### Challenge 3 (Hard): Share Result Button

In [ ]:
# Currently result URLs expire after 1 hour.
# Challenge: add a 'Share' button that copies the result URL to clipboard.

share_button_js = """
// Add this to results.html
<button onclick="copyLink()" class="bg-gray-700 hover:bg-gray-600 text-white px-4 py-2 rounded-lg">
    🔗 Copy Link
</button>

<script>
function copyLink() {
    navigator.clipboard.writeText(window.location.href);
    alert('Link copied! Note: expires in 1 hour.');
}
</script>
"""
print(share_button_js)

### Challenge 4 (Stretch): Webcam Mode

The `detect.html` template already has a webcam tab with `static/js/webcam.js`.  
If you haven't tested it yet:

1. Visit your live app on `https://` (webcam requires HTTPS)
2. Go to Detect → click "Use Webcam" tab
3. Allow camera access
4. Click "Capture" then "Detect My Mood"

Gemini analyses the image and detects your facial expression!

---
## ✅ Course Complete! 🎉

**In 8 sessions, starting from zero, you built and deployed:**

- A full-stack Python web application
- AI-powered emotion detection using Google Gemini
- Music recommendations with real Spotify/YouTube links
- User sessions and a cloud database (Supabase)
- 5 distinct pages: landing, detect, results, history, profile
- Deployed live to the internet on Render

**What you learned:**
- Python (variables, functions, lists, dicts, loops, classes)
- APIs and JSON
- Web development (HTTP, FastAPI, Jinja2, HTML, CSS)
- Database basics (Supabase/PostgreSQL)
- Git version control and deployment

**Next steps:**
- Add your app to your LinkedIn and GitHub profile
- Explore the [FastAPI docs](https://fastapi.tiangolo.com)
- Try the [Gemini API cookbook](https://github.com/google-gemini/cookbook)
- Build your next project!